In [31]:
import requests
import time

In [32]:

def fetch_page(base_url, params=None, delay=1):
    """Fetch a single web page politely. Returns HTML text, or None on failure."""
    headers = {
        "User-Agent": "PoetryResearch (au789885@uni.au.dk)"
    }
    try:
        response = requests.get(base_url, params=params,
                                headers=headers, timeout=10)
        response.raise_for_status()
        time.sleep(delay)          # pause after a successful fetch
        return response.text

    except requests.exceptions.RequestException as e:
        print(f"Failed to fetch {base_url} ({params}): {e}")
        return None
    


In [33]:
poets = fetch_page("https://poets.org/poems", params={"page": 1})
print("poets.org ->", "got", len(poets), "chars" if poets else "nothing")

pf = fetch_page("https://www.poetryfoundation.org/poems/browse",
                params={"sortBy": "prod_PFONE_post_date_desc", "page": 0, "query": ""})
print("poetryfoundation.org ->", "got", len(pf), "chars" if pf else "nothing")


poets.org -> got 74471 chars
poetryfoundation.org -> got 310776 chars


In [34]:
from bs4 import BeautifulSoup
poem_links = []
base_url = "https://www.poets.org"
for page_number in range(2):       
    html = fetch_page("https://www.poets.org/poems", params={"page": page_number})
    soup = BeautifulSoup(html, "html.parser")
    links = soup.find_all("a")
    new_links = [
        base_url + a["href"]
        for a in soup.find_all("a", href=True)
        if a["href"].startswith("/poem/")
    ]
    print(f"Collected page {page_number} ({len(poem_links)} chars)")
    poem_links.extend(new_links)



Collected page 0 (0 chars)
Collected page 1 (20 chars)


In [35]:
poems = []

for url in poem_links:
    new_poems = fetch_page(url, params=None)
    poems.append(new_poems)

In [28]:
import re

def parse_poems(html):
    soup = BeautifulSoup(html, "html.parser")

    # TITLE
    title_tag = soup.find("h1")
    poem_title = title_tag.get_text(strip=True) if title_tag else None

    # POEM TEXT
    body_tag = soup.find("div", class_="field--body")
    poem_text = body_tag.get_text(strip=True) if body_tag else None

    # ORIGINAL YEAR
    pub_date = soup.find("div", class_="field field--field_date_published")
    pub_text = pub_date.get_text(strip=True) if pub_date else None
    
    # Extract a 4-digit year from the text, then convert to int
    publication_year = 0
    if pub_text:
        match = re.search(r"\d{4}", pub_text)
        if match:
            publication_year = int(match.group())
    
    # AUTHOR
    author_tag = soup.find("a", href=lambda x: x and "/poet/" in x)
    author = author_tag.get_text(strip=True) if author_tag else None
    
    if publication_year is not None and publication_year > 1999:
        return {
            "title": poem_title,
            "text": poem_text,
            "published": publication_year,
            "author": author,
        }
    
parsed = []
for poem in poems:
    new_parsing = parse_poems(poem)
    parsed.append(new_parsing)



In [10]:
print(parsed[1])

None


In [29]:
import uuid
poems = [p for p in parsed if p is not None]

poems_by_id = {f"P{i:04d}": p for i, p in enumerate(poems, start=1000)}

print(poems_by_id["P1001"])

{'title': 'The Hollow Men', 'text': 'A penny for the Old GuyIWe are the hollow menWe are the stuffed menLeaning togetherHeadpiece filled with straw. Alas!Our dried voices, whenWe whisper togetherAre quiet and meaninglessAs wind in dry grassOr rats’ feet over broken glassIn our dry cellarShape without form, shade without colour.Paralysed force, gesture without motion;Those who have crossedWith direct eyes, to death’s other KingdomRemember us—if at all—not as lostViolent souls, but onlyAs the hollow menIIEyes I dare not meet in dreamsIn death’s dream kingdomThese do not appear:There, the eyes areSunlight on a broken columnThere, is a tree swingingAnd voices areIn the wind’s singingMore distant and more solemnThan a fading star.Let me be no nearerIn death’s dream kingdomLet me also wearSuch deliberate disguisesRat’s coat, crowskin, crossed stavesIn a fieldBehaving as the wind behavesNo nearer—Not that final meetingIn the twilight kingdomIIIThis is the dead landThis is cactus landHere the 

In [30]:
import pandas as pd
import re

df = pd.DataFrame.from_dict(poems_by_id, orient="index")

for col in df.select_dtypes(include="object").columns:

    df[col] = df[col].str.replace(r'([.!?,;:"])(?=[A-Za-z])', r'\1 ', regex=True)

    df[col] = df[col].str.replace(r'(?<=[a-z])(?=[A-Z])', ' ', regex=True)

df.to_csv("poems101.csv", index=True, encoding="utf-8")

C:\Users\Sipos Dorottya\AppData\Local\Temp\ipykernel_22696\1178358393.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:
